Task 1: generating individual images

In [ ]:
from numpy import array, diagonal, shape
from scipy.linalg import eig
from matplotlib import pyplot as plt

def A(p):
    return array([[     5, 0*p,  0*p, -1*p],
                  [   1*p,   0, -1*p,  1*p],
                  [-1.5*p, 1*p,   -2,  1*p],
                  [  -1*p, 1*p,  3*p,   -3]])

def gersh(A):
    # returns list of real number pairs (center, radius) of the disks
    list = []
    for i in range(shape(A)[0]):
        list.append((A[i,i], sum(abs(A)[i,:])-abs(A)[i,i]))
    return list

F = 101

for p in range(F):
    fig = plt.figure()
    ax = plt.axes(xlim=(-9, 7), ylim=(-6, 6))
    Ap = A(p/(F-1))
    d = diagonal(Ap)
    e = eig(Ap)[0]
    for center, radius in gersh(Ap):
        ax.add_patch(plt.Circle((center, 0), radius, color='orange', alpha=0.3))
    ax.scatter(e.real, e.imag, 20, label='Eigenvalues')
    ax.scatter(d.real, d.imag, 5, label='Diagonal elements')
    plt.legend()
    plt.title(f"Gershgorin disks, p={(p/(F-1)):.3f}", fontname='DejaVu Sans Mono')
    plt.savefig(f'images/{int(p)}.png', dpi=300) # an 'images' directory must exist


Task 1: combining the images into a video

In [ ]:
import cv2

F = 101
fps = 50
files_and_duration = [(f'images/{i}.png', 0.04 if i!=F-1 else 3) for i in range(F)]

h, w, _ = cv2.imread('images/0.png').shape
fourcc = cv2.VideoWriter_fourcc('m', 'p', '4', 'v')
writer = cv2.VideoWriter('output.mp4', fourcc, fps, (w, h))

for file, duration in files_and_duration:
    frame = cv2.imread(file)
    for repeat in range(int(duration * fps)):
        writer.write(frame)

writer.release()

Task 2: introducing a tridiag function for triangulization

In [ ]:
from numpy import zeros, shape, identity, array, append, diagonal
from numpy.random import rand
from scipy.linalg import eig, norm, qr

def tridiag(A):
    m = shape(A)[0]
    for k in range(m-2):
        x = A[k:,k].reshape(m-k,1)
        y = zeros(m-k).reshape(m-k,1)
        y[0] = x[0]
        y[1] = norm(x[1:])
        w = (x-y)/norm(x-y)
        P = identity(m-k) - 2*w@w.T
        A[k:,k:] = P@A[k:,k:]@P
    return A

Task 2: introducing the practical function for the "practical QR algorithm"

In [ ]:
iteration_cap = 1

def practical(A,it=1):
    global iteration_cap
    m = shape(A)[0]
    if m == 0:
        return array([])
    
    shift = A[-1,-1]*identity(m)
    Q, R = qr(A-shift)
    A = R@Q+shift

    if it == iteration_cap:
        return diagonal(A)
    
    res = array([])
    f = 0
    for j in range(1,m):
        if abs(A[j-1,j]) < 10e-8:
            res = append(res, practical(A[f:j,f:j], it+1))
            f = j
    res = append(res, practical(A[f:,f:], it+1))
    return res

Task 2: plotting the behavior of random symmetric matrices

In [ ]:
import matplotlib.pyplot as plt 

matrices = [rand(k,k) for k in range(20,101,20)]
for k in range(len(matrices)):
    matrices[k] = (matrices[k] + matrices[k].T) / 2
iteration_cap = 100
plt.plot([20,100],[1e-8,1e-8],linestyle='dashed',color='black')
for t in [50,100,150,200,250]:
    iteration_cap = t
    k = 20
    k_list = []
    i_list = []
    for A in matrices:
        e = eig(A)[0]
        e.sort()
        X = tridiag(A)
        p = practical(X)
        p.sort()
        k_list.append(k)
        i_list.append(max([abs(e[i] - p[i]) for i in range(k)]))
        k += 20
    plt.plot(k_list, i_list, label=f"{t}")
plt.xticks([20,40,60,80,100], ["20x20", "40x40", "60x60", "80x80", "100x100"])
plt.yscale('log')
plt.title(r"Error when comparing $practical$ and $eig$")
plt.xlabel("Dimensions")
plt.ylabel("Error")
plt.legend(title=r"Iterations of $practical$")
plt.savefig("iterations.png", dpi=300)

Task 2: testing a specific orthogonal symmetric matrix

In [ ]:
from numpy import set_printoptions
set_printoptions(precision=8,suppress=True)

Qu = array([[2.,-2.,1.],[1.,2.,2.],[2.,1.,-2.]])
Dg = array([[4.,0.,0.],[0.,2.,0.],[0.,0.,7.]])
Qu /= norm(Qu, ord=2)
Dg /= norm(Dg, ord=2)
Qs = Qu.T@Dg@Qu
iteration_cap = 8
print("Eigenvalues of an orthogonal symmetric matrix")
print("      eig:", eig(Qs)[0].real)
print("practical:", practical(tridiag(Qs)))

Task 3: finding an upper-Hessenberg matrix with repeating eigenvalues

In [ ]:
from numpy.random import randint
from numpy import unique
while True:
    H = randint(1,5,(4,4))
    H[2,0] = 0
    H[3,0] = 0
    H[3,1] = 0
    e = eig(H)[0]
    if len(unique(e)) != len(e):
        print(H)
        print(eig(H)[0])
        break